In [1]:
import torch
import gc, re, os, hashlib
import numpy as np
from scipy.special import softmax
from jax.tree_util import tree_map
from esmfold import ESMFold

In [2]:
## FOR OUTPUT VISUALIZATION

def parse_output(output):
    pae = (output["aligned_confidence_probs"][0] * np.arange(64)).mean(-1) * 31
    plddt = output["plddt"][0,:,1]
    
    bins = np.append(0,np.linspace(2.3125,21.6875,63))
    sm_contacts = softmax(output["distogram_logits"],-1)[0]
    sm_contacts = sm_contacts[...,bins<8].sum(-1)
    xyz = output["positions"][-1,0,:,1]
    mask = output["atom37_atom_exists"][0,:,1] == 1
    o = {"pae":pae[mask,:][:,mask],
         "plddt":plddt[mask],
         "sm_contacts":sm_contacts[mask,:][:,mask],
         "xyz":xyz[mask]}
    return o

In [3]:
## GARBAGE COLLECTION

gc.collect()
torch.cuda.empty_cache()

In [4]:
## LOAD MODEL AND TRANSFER WEIGHTS
%load_ext autoreload
%autoreload 2

pretrained = torch.load("esmfold.model", weights_only=False)
model = ESMFold(esmfold_config=pretrained.cfg) ## make copy of model with new forward pass and then transfer the weights
model.load_state_dict(pretrained.state_dict())

model.eval().cuda().requires_grad_(False)

del pretrained
gc.collect()
torch.cuda.empty_cache()

In [5]:
## SHORT INPUT EXAMPLE

short_sequence = "GWSTELEKHREELKEFLKKEGITNVEIRIDNGRLEVRVEGGTERLKRFLEELRQKLEKKGYTVDIKIE" 
short_sequence = re.sub("[^A-Z:]", "", short_sequence.replace("/",":").upper())
short_sequence = re.sub(":+",":",short_sequence)
short_sequence = re.sub("^[:]+","",short_sequence)
short_sequence = re.sub("[:]+$","",short_sequence)
copies = 1 
short_sequence = ":".join([short_sequence] * copies)
num_recycles = 3 #["0", "1", "2", "3", "6", "12", "24"]
chain_linker = 25

In [6]:
## FORWARD PASS

def get_hash(x): return hashlib.sha1(x.encode()).hexdigest()
ID ="test_"+get_hash(short_sequence)[:5]
output = model.infer(short_sequence,
                     num_recycles=num_recycles,
                     residue_index_offset=512)

pdb_str = model.output_to_pdb(output)[0]
output = tree_map(lambda x: x.cpu().numpy(), output)
ptm = output["ptm"][0]
plddt = output["plddt"][0,...,1].mean()
O = parse_output(output)
print(f'ptm: {ptm:.3f} plddt: {plddt:.3f}')
os.system(f"mkdir -p {ID}")
prefix = f"{ID}/ptm{ptm:.3f}_r{num_recycles}_default"
np.savetxt(f"{prefix}.pae.txt",O["pae"],"%.3f")
with open(f"{prefix}.pdb","w") as out:
    out.write(pdb_str)
    
del output, pdb_str, O
gc.collect()
torch.cuda.empty_cache()

esm2_encoding 158.9155262336135
esm_preprocessing 32.07354433834553
folding_trunk_total 1676.5051437541842
prediction_heads 14.25624918192625
total_forward 1947.5469021126628
ptm: 0.817 plddt: 90.519


In [7]:
## LONG INPUT EXAMPLE

long_sequence = "MGFLKLIEIENFKSYKGRQIIGPFQRFTAIIGPNGSGKSNLMDAISFVLGEKTSNLRVKTLRDLIHGAPVGKPAANRAFVSMVYSEEGAEDRTFARVIVGGSSEYKINNKVVQLHEYSEELEKLGILIKARNFLVFQGAVESIAMKNPKERTALFEEISRSGELAQEYDKRKKEMVKAEEDTQFNYHRKKNIAAERKEAKQEKEEADRYQRLKDEVVRAQVQLQLFKLYHNEVEIEKLNKELASKNKEIEKDKKRMDKVEDELKEKKKELGKMMREQQQIEKEIKEKDSELNQKRPQYIKAKENTSHKIKKLEAAKKSLQNAQKHYKKRKGDMDELEKEMLSVEKARQEFEERMEEESQSQGRDLTLEENQVKKYHRLKEEASKRAATLAQELEKFNRDQKADQDRLDLEERKKVETEAKIKQKLREIEENQKRIEKLEEYITTSKQSLEEQKKLEGELTEEVEMAKRRIDEINKELNQVMEQLGDARIDRQESSRQQRKAEIMESIKRLYPGSVYGRLIDLCQPTQKKYQIAVTKVLGKNMDAIIVDSEKTGRDCIQYIKEQRGEPETFLPLDYLEVKPTDEKLRELKGAKLVIDVIRYEPPHIKKALQYACGNALVCDNVEDARRIAFGGHQRHKTVALDGTLFQKSGVISGGASDLKAKARRWDEKAVDKLKEKKERLTEELKEQMKAKRKEAELRQVQSQAHGLQMRLKYSQSDLEQTKTRHLALNLQEKSKLESELANFGPRINDIKRIIQSREREMKDLKEKMNQVEDEVFEEFCREIGVRNIREFEEEKVKRQNEIAKKRLEFENQKTRLGIQLDFEKNQLKEDQDKVHMWEQTVKKDENEIEKLKKEEQRHMKIIDETMAQLQDLKNQHLAKKSEVNDKNHEMEEIRKKLGGANKEMTHLQKEVTAIETKLEQKRSDRHNLLQACKMQDIKLPLSKGTMDDISQEEGSSQGEDSVSGSQRISSIYAREALIEIDYGDLCEDLKDAQAEEEIKQEMNTLQQKLNEQQSVLQRIAAPNMKAMEKLESVRDKFQETSDEFEAARKRAKKAKQAFEQIKKERFDRFNACFESVATNIDEIYKALSRNSSAQAFLGPENPEEPYLDGINYNCVAPGKRFRPMDNLSGGEKTVAALALLFAIHSYKPAPFFVLDEIDAALDNTNIGKVANYIKEQSTCNFQAIVISLKEEFYTKAESLIGVYPEQGDCVISKVLTFDLTKYPDANPNPNEQ" 
long_sequence = re.sub("[^A-Z:]", "", long_sequence.replace("/",":").upper())
long_sequence = re.sub(":+",":",long_sequence)
long_sequence = re.sub("^[:]+","",long_sequence)
long_sequence = re.sub("[:]+$","",long_sequence)
copies = 1 
long_sequence = ":".join([long_sequence] * copies)
num_recycles = 3 #["0", "1", "2", "3", "6", "12", "24"]
chain_linker = 25

In [8]:
## FORWARD PASS

def get_hash(x): return hashlib.sha1(x.encode()).hexdigest()
ID ="test_"+get_hash(long_sequence)[:5]
output = model.infer(long_sequence,
                     num_recycles=num_recycles,
                     residue_index_offset=512)

pdb_str = model.output_to_pdb(output)[0]
output = tree_map(lambda x: x.cpu().numpy(), output)
ptm = output["ptm"][0]
plddt = output["plddt"][0,...,1].mean()
O = parse_output(output)
print(f'ptm: {ptm:.3f} plddt: {plddt:.3f}')
os.system(f"mkdir -p {ID}")
prefix = f"{ID}/ptm{ptm:.3f}_r{num_recycles}_default"
np.savetxt(f"{prefix}.pae.txt",O["pae"],"%.3f")
with open(f"{prefix}.pdb","w") as out:
    out.write(pdb_str)
del output, pdb_str, O
gc.collect()
torch.cuda.empty_cache()

esm2_encoding 155.4422527551651
esm_preprocessing 1.9820397719740868
folding_trunk_total 267.42903888225555
total_forward 425.47585908323526


OutOfMemoryError: CUDA out of memory. Tried to allocate 27.93 GiB. GPU 0 has a total capacity of 23.52 GiB of which 6.95 GiB is free. Process 3981183 has 16.56 GiB memory in use. Of the allocated memory 15.58 GiB is allocated by PyTorch, and 540.67 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
## (OPTIONAL) VISUALIZATION OF PREDICTION

import py3Dmol
lengths = [len(s) for s in long_sequence.split(":")]
pymol_color_list = ["#33ff33","#00ffff","#ff33cc","#ffff00","#ff9999","#e5e5e5","#7f7fff","#ff7f00",
                    "#7fff7f","#199999","#ff007f","#ffdd5e","#8c3f99","#b2b2b2","#007fff","#c4b200",
                    "#8cb266","#00bfbf","#b27f7f","#fcd1a5","#ff7f7f","#ffbfdd","#7fffff","#ffff7f",
                    "#00ff7f","#337fcc","#d8337f","#bfff3f","#ff7fff","#d8d8ff","#3fffbf","#b78c4c",
                    "#339933","#66b2b2","#ba8c84","#84bf00","#b24c66","#7f7f7f","#3f3fa5","#a5512b"]

def show_pdb(pdb_str, show_sidechains=False, show_mainchains=False,
             color="pLDDT", chains=None, vmin=50, vmax=90,
             size=(800,480), hbondCutoff=4.0,
             Ls=None,
             animate=False):

  if chains is None:
    chains = 1 if Ls is None else len(Ls)
  view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js', width=size[0], height=size[1])
  if animate:
    view.addModelsAsFrames(pdb_str,'pdb',{'hbondCutoff':hbondCutoff})
  else:
    view.addModel(pdb_str,'pdb',{'hbondCutoff':hbondCutoff})
  if color == "pLDDT":
    view.setStyle({'cartoon': {'colorscheme': {'prop':'b','gradient': 'roygb','min':vmin,'max':vmax}}})
  elif color == "rainbow":
    view.setStyle({'cartoon': {'color':'spectrum'}})
  elif color == "chain":
    for n,chain,color in zip(range(chains),alphabet_list,pymol_color_list):
       view.setStyle({'chain':chain},{'cartoon': {'color':color}})
  if show_sidechains:
    BB = ['C','O','N']
    view.addStyle({'and':[{'resn':["GLY","PRO"],'invert':True},{'atom':BB,'invert':True}]},
                  {'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
    view.addStyle({'and':[{'resn':"GLY"},{'atom':'CA'}]},
                  {'sphere':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
    view.addStyle({'and':[{'resn':"PRO"},{'atom':['C','O'],'invert':True}]},
                  {'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
  if show_mainchains:
    BB = ['C','O','N','CA']
    view.addStyle({'atom':BB},{'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
  view.zoomTo()
  if animate: view.animate()
  return view

color = "confidence" #@param ["confidence", "rainbow", "chain"]
if color == "confidence": color = "pLDDT"
show_sidechains = False #@param {type:"boolean"}
show_mainchains = False #@param {type:"boolean"}
show_pdb(pdb_str, color=color,
         show_sidechains=show_sidechains,
         show_mainchains=show_mainchains,
         Ls=lengths).show()